# APIM ❤️ Microsoft Purview

## Microsoft Purview DLP at the AI Gateway lab
![flow](../../images/apim-purview-dlp.png)

Playground to enforce [Microsoft Purview](https://learn.microsoft.com/purview/purview) Data Loss Prevention on prompts **and** responses at the AI gateway using the [Purview `processContent` API](https://learn.microsoft.com/graph/api/userprotectionscopecontainer-computeinheritance). The gateway calls Purview twice per turn — once on the inbound prompt (Gate 1, `uploadText`) and once on the outbound response (Gate 3, `downloadText`) — via a shared APIM policy fragment applied identically to two backends: **Microsoft Foundry** (gpt-4.1) and **Amazon Bedrock** (Nova 2 Lite, signed with SigV4 in the APIM policy, no Lambda).

[View Foundry policy](foundry-hrpolicy.policy.xml) · [View Bedrock policy](bedrock-expense.policy.xml) · [View shared Purview fragment](purview-processcontent.fragment.xml) · [View content-log fragment](emit-content-log.fragment.xml)

### Prerequisites

Complete **all** of the [README prerequisites](README.MD) before running this notebook. The Microsoft Entra app registration + admin consent + Purview collection & DLP policy configuration cannot be done from the notebook; they require tenant-level permissions and are shared with other workloads. You will paste the resulting `client id`, `client secret`, and AWS access keys into the initialization cell below.

> ⚠️ **All cost, pricing, and dollar-value figures shown in this lab — including the KQL in `aigw-payg-purview-forecast.kql` and any inline comments referencing per-token or per-call prices — are illustrative only.** They exist to explain *how* the telemetry is stitched together, not *what* it costs. Confirm authoritative pricing with your Microsoft account team or an approved Microsoft sales channel before quoting any number against a real workload.

▶️ Click `Run All` to execute all steps sequentially, or execute them `Step by Step`...

<a id='0'></a>
### 0️⃣ Initialize notebook variables

- Resources will be suffixed by a unique string based on your subscription id.
- Adjust the location parameter based on [product availability by Azure region](https://azure.microsoft.com/explore/global-infrastructure/products-by-region/?products=api-management).
- Paste values obtained during the [README prerequisites](README.MD): the gateway app registration's client id + secret, your AWS Bedrock access key + secret, and your Foundry endpoint + agent name.
- Client requests only carry `{"input": "..."}` — the model / agent name is injected server-side by the Foundry policy from the `foundry-agent-name` named value (which is set from `foundry_agent_name` below), and the Bedrock model id is injected the same way from `bedrock-model-id`. Callers never have to know the deployment name.
- Leave `content_log_reader_principal_id` empty to skip the ABAC-scoped Log Analytics Data Reader assignment; the `AIGatewayContent_CL` table will still be created but only workspace admins can query it.

In [ ]:
import os, sys, json, uuid
sys.path.insert(1, '../../shared')  # add the shared directory to the Python path
import utils

deployment_name       = os.path.basename(os.path.dirname(globals()['__vsc_ipynb_file__']))
resource_group_name   = f"lab-{deployment_name}"
resource_group_location = "eastus2"

apim_sku                  = 'Standardv2'
apim_subscriptions_config = [{"name": "subscription1", "displayName": "Subscription 1"}]

# --- Foundry backend -----------------------------------------------------------
foundry_project_endpoint = ''   # e.g. 'myfoundry.services.ai.azure.com/api/projects/myProject' (no scheme, no trailing slash)\n
foundry_agent_name       = ''   # e.g. 'hrpolicy-agent'

# --- Amazon Bedrock backend ---------------------------------------------------
bedrock_region       = 'us-east-1'
bedrock_model_id     = 'us.amazon.nova-2-lite-v1:0'
aws_bedrock_access_key = ''   # IAM access key id with bedrock:InvokeModel permission
aws_bedrock_secret_key = ''   # IAM secret access key. Stored as APIM secure named value.

# --- Gateway service principal + Purview --------------------------------------
graph_tenant_id     = ''   # az account show --query tenantId -o tsv
graph_client_id     = ''   # app registration client id from README prerequisites
graph_client_secret = ''   # app registration client secret. Stored as APIM secure named value.
purview_graph_host  = 'graph.microsoft.com'

# --- Granular RBAC on AIGatewayContent_CL -------------------------------------
content_log_reader_principal_id   = ''   # az ad user show --id <upn> --query id -o tsv
content_log_reader_principal_type = 'User'

utils.print_ok('Notebook initialized')

<a id='1'></a>
### 1️⃣ Verify the Azure CLI and the connected Azure subscription

In [ ]:
output = utils.run("az account show", "Retrieved az account", "Failed to get the current az account")

if output.success and output.json_data:
    current_user   = output.json_data['user']['name']
    tenant_id      = output.json_data['tenantId']
    subscription_id = output.json_data['id']

    utils.print_info(f"Current user: {current_user}")
    utils.print_info(f"Tenant ID: {tenant_id}")
    utils.print_info(f"Subscription ID: {subscription_id}")

    # Default graph_tenant_id from the signed-in tenant if the user left it blank.
    if not graph_tenant_id:
        graph_tenant_id = tenant_id
        utils.print_info(f"graph_tenant_id defaulted to signed-in tenant: {graph_tenant_id}")

<a id='2'></a>
### 2️⃣ Create deployment using 🦾 Bicep

This lab uses [Bicep](https://learn.microsoft.com/azure/azure-resource-manager/bicep/overview?tabs=bicep) to declaratively define the resources deployed in the specified resource group. Change the parameters or the [main.bicep](main.bicep) directly to try different configurations.

In [ ]:
# Sanity-check the prereq values before deploying
required = {
    'foundry_project_endpoint': foundry_project_endpoint,
    'foundry_agent_name':       foundry_agent_name,
    'aws_bedrock_access_key':   aws_bedrock_access_key,
    'aws_bedrock_secret_key':   aws_bedrock_secret_key,
    'graph_client_id':          graph_client_id,
    'graph_client_secret':      graph_client_secret,
    'graph_tenant_id':          graph_tenant_id,
}
missing = [k for k, v in required.items() if not v]
if missing:
    utils.print_error(f"Missing required values: {missing}. Fill them in Step 0 before running Step 2.")
    raise SystemExit(1)

utils.create_resource_group(resource_group_name, resource_group_location)

bicep_parameters = {
    "$schema": "https://schema.management.azure.com/schemas/2019-04-01/deploymentParameters.json#",
    "contentVersion": "1.0.0.0",
    "parameters": {
        "apimSku":                        { "value": apim_sku },
        "apimSubscriptionsConfig":        { "value": apim_subscriptions_config },
        "foundryProjectEndpoint":         { "value": foundry_project_endpoint },
        "foundryAgentName":               { "value": foundry_agent_name },
        "bedrockRegion":                  { "value": bedrock_region },
        "bedrockModelId":                 { "value": bedrock_model_id },
        "awsBedrockAccessKey":            { "value": aws_bedrock_access_key },
        "awsBedrockSecretKey":            { "value": aws_bedrock_secret_key },
        "graphTenantId":                  { "value": graph_tenant_id },
        "graphClientId":                  { "value": graph_client_id },
        "graphClientSecret":              { "value": graph_client_secret },
        "purviewGraphHost":               { "value": purview_graph_host },
        "contentLogReaderPrincipalId":    { "value": content_log_reader_principal_id },
        "contentLogReaderPrincipalType":  { "value": content_log_reader_principal_type }
    }
}

with open('params.json', 'w') as bicep_parameters_file:
    bicep_parameters_file.write(json.dumps(bicep_parameters))

output = utils.run(
    f"az deployment group create --name {deployment_name} --resource-group {resource_group_name} --template-file main.bicep --parameters params.json",
    f"Deployment '{deployment_name}' succeeded",
    f"Deployment '{deployment_name}' failed"
)

<a id='3'></a>
### 3️⃣ Get the deployment outputs and the APIM subscription key

In [ ]:
output = utils.run(
    f"az deployment group show --name {deployment_name} -g {resource_group_name}",
    f"Retrieved deployment: {deployment_name}",
    f"Failed to retrieve deployment: {deployment_name}"
)

if output.success and output.json_data:
    apim_gateway_url        = utils.get_deployment_output(output, 'apimGatewayUrl',        'APIM gateway URL')
    foundry_api_path        = utils.get_deployment_output(output, 'foundryApiPath',        'Foundry API path')
    bedrock_api_path        = utils.get_deployment_output(output, 'bedrockApiPath',        'Bedrock API path')
    log_analytics_id        = utils.get_deployment_output(output, 'logAnalyticsWorkspaceId','Log Analytics Id')

    foundry_endpoint = f"{apim_gateway_url}/{foundry_api_path}"
    bedrock_endpoint = f"{apim_gateway_url}/{bedrock_api_path}"

# Pull the APIM subscription key created by the apimSubscriptionsConfig array.
apim_resource = f"apim-{output.json_data['properties']['outputs']['apimServiceId']['value'].split('/')[-1] if False else ''}"
keys = utils.run(
    f"az apim subscription list --resource-group {resource_group_name} --service-name apim-{utils.get_resource_suffix(subscription_id, resource_group_name)} --query \"[?name=='subscription1'].{{name:name}}\" -o tsv",
    "Listed APIM subscriptions",
    "Failed to list APIM subscriptions"
) if hasattr(utils, 'get_resource_suffix') else None

# Simpler: use az rest to grab the primary key of the 'subscription1' subscription
import subprocess
apim_name_lookup = utils.run(
    f"az resource list --resource-group {resource_group_name} --resource-type Microsoft.ApiManagement/service --query \"[0].name\" -o tsv",
    "Located APIM instance",
    "Failed to locate APIM instance"
)
apim_name = (apim_name_lookup.text or '').strip() if apim_name_lookup and apim_name_lookup.success else None

apim_subscription_key = None
if apim_name:
    sub_keys = utils.run(
        f"az apim subscription show --resource-group {resource_group_name} --service-name {apim_name} --sid subscription1",
        f"Retrieved subscription key for '{apim_name}'",
        f"Failed to retrieve subscription key for '{apim_name}'"
    )
    if sub_keys and sub_keys.success and sub_keys.json_data:
        apim_subscription_key = sub_keys.json_data.get('primaryKey')

utils.print_info(f"APIM gateway URL:  {apim_gateway_url}")
utils.print_info(f"Foundry endpoint:  {foundry_endpoint}")
utils.print_info(f"Bedrock endpoint:  {bedrock_endpoint}")
utils.print_info(f"Subscription key : {'*' * 8 + (apim_subscription_key[-4:] if apim_subscription_key else 'MISSING')}")

<a id='4'></a>
### 4️⃣ Acquire a user access token for the OBO exchange

The gateway policy expects an `X-User-Token` header containing the **signed-in user's** Entra access token, audience-scoped to the gateway app registration. APIM performs the on-behalf-of exchange against Microsoft Graph inside the policy and caches the resulting Graph token for the rest of the turn.

If the following cell returns 401 or `AADSTS500011`, verify:

1. The app registration created in the README prerequisites has an **Expose an API** scope (`user_impersonation`) and a default scope URI matching `api://<graph_client_id>`.
2. The signed-in `az` account has consented to the app (`az login --scope api://<graph_client_id>/.default` at least once).
3. Admin consent has been granted for the delegated Microsoft Graph permissions (`ProtectionScopes.Compute.User`, `ContentActivity.Write`).

In [ ]:
token_out = utils.run(
    f"az account get-access-token --resource api://{graph_client_id} --query accessToken -o tsv",
    "Acquired user token for gateway app",
    "Failed to acquire user token \u2014 see the note above"
)
user_token = (token_out.text or '').strip() if token_out and token_out.success else None
if not user_token:
    utils.print_error('X-User-Token is empty. Fix the app registration setup and re-run this cell before continuing.')
else:
    utils.print_info(f"User token length: {len(user_token)} characters (JWT)")

<a id='5'></a>
### 5️⃣ Test: Microsoft Foundry — happy path

A benign prompt should pass **Gate 1** (`uploadText`), be forwarded to Foundry, come back through **Gate 3** (`downloadText`), and return HTTP 200 with `X-Purview-Prompt-Calls: 1`, `X-Purview-Response-Calls: 1`, and `X-Purview-Blocked: 0`.

In [ ]:
import requests

correlation_id = str(uuid.uuid4())
resp = requests.post(
    f"{foundry_endpoint}/chat",
    headers={
        'Ocp-Apim-Subscription-Key': apim_subscription_key,
        'X-User-Token':              user_token,
        'X-Correlation-Id':          correlation_id,
        'Content-Type':              'application/json'
    },
    json={
        'input': 'Summarize our vacation policy in one paragraph.'
    },
    timeout=60
)
utils.print_info(f"HTTP {resp.status_code}")
for h in ('X-Purview-Prompt-Calls', 'X-Purview-Response-Calls', 'X-Purview-Blocked', 'X-Purview-Block-Reason', 'X-Model-Input-Tokens', 'X-Model-Output-Tokens', 'X-Correlation-Id'):
    utils.print_info(f"  {h}: {resp.headers.get(h)}")
print(resp.text[:800])

<a id='6'></a>
### 6️⃣ Test: Microsoft Foundry — Gate 1 (prompt) block

A prompt containing content matched by your Purview DLP policy (for example, a fake credit-card number or SSN pattern) should be **blocked at Gate 1** — the model is never called. Expect HTTP 403 with `X-Purview-Blocked: 1` and `X-Purview-Block-Reason: PURVIEW_DLP_PROMPT`.

The exact string that trips the block depends on the DLP policy configured in your Purview tenant. Replace the placeholder below with something you *know* your policy will match. **Never paste a real PII value.**

In [ ]:
correlation_id = str(uuid.uuid4())
resp = requests.post(
    f"{foundry_endpoint}/chat",
    headers={
        'Ocp-Apim-Subscription-Key': apim_subscription_key,
        'X-User-Token':              user_token,
        'X-Correlation-Id':          correlation_id,
        'Content-Type':              'application/json'
    },
    json={
        # Fake credit card number, format only. Replace with a sensitive-info marker your Purview policy will block.
        'input': 'Log this test card: 4111-1111-1111-1111 for reimbursement.'
    },
    timeout=60
)
utils.print_info(f"HTTP {resp.status_code} (expect 403)")
for h in ('X-Purview-Prompt-Calls', 'X-Purview-Response-Calls', 'X-Purview-Blocked', 'X-Purview-Block-Reason', 'X-Correlation-Id'):
    utils.print_info(f"  {h}: {resp.headers.get(h)}")
print(resp.text[:600])

<a id='6b'></a>
### 6️⃣.5 Test: Microsoft Foundry — Gate 3 (response) block

Gate 3 (`downloadText`) inspects the model *response*. Because production models refuse to emit real PII, the shipped policy includes a **demo simulator**: if the user prompt contains the phrase `stripe test credit card`, the outbound policy replaces the model's response text with a fixed string containing a Luhn-valid Visa **before** Purview scans it. Purview then blocks with `PURVIEW_DLP_RESPONSE`. This makes the response-lane demo reproducible without asking the model for sensitive data.

> Simulator triggers (case-insensitive):
> - `stripe test credit card` → forces a Credit Card Number SIT match
> - `fictional hr documentation` → forces a U.S. SSN SIT match
>
> The simulator lives in the two policy files under section `4-sim` (Foundry) and `6-sim` (Bedrock). Remove those `<choose>` blocks for production use.


In [ ]:
correlation_id = str(uuid.uuid4())
resp = requests.post(
    f"{foundry_endpoint}/chat",
    headers={
        'Ocp-Apim-Subscription-Key': apim_subscription_key,
        'X-User-Token':              user_token,
        'X-Correlation-Id':          correlation_id,
        'Content-Type':              'application/json'
    },
    json={
        # Demo trigger: the policy simulator will replace the model's text with
        # a canned Stripe test card BEFORE Purview scans, guaranteeing a block.
        'input': 'What is a stripe test credit card and how is it used in developer sandboxes?'
    },
    timeout=60
)
utils.print_info(f"HTTP {resp.status_code} (expect 403)")
for h in ('X-Purview-Prompt-Calls', 'X-Purview-Response-Calls', 'X-Purview-Blocked', 'X-Purview-Block-Reason', 'X-Correlation-Id'):
    utils.print_info(f"  {h}: {resp.headers.get(h)}")
print(resp.text[:600])

<a id='7'></a>
### 7️⃣ Test: Amazon Bedrock — happy path

Same benign prompt as Step 5, but routed to the Bedrock backend. APIM signs the request with SigV4 inside the policy and forwards it to `bedrock-runtime.<region>.amazonaws.com/model/<modelId>/invoke`. Response headers use the same wire contract as the Foundry route.

In [ ]:
correlation_id = str(uuid.uuid4())
resp = requests.post(
    f"{bedrock_endpoint}/chat",
    headers={
        'Ocp-Apim-Subscription-Key': apim_subscription_key,
        'X-User-Token':              user_token,
        'X-Correlation-Id':          correlation_id,
        'Content-Type':              'application/json'
    },
    json={
        'input': 'Summarize our vacation policy in one paragraph.'
    },
    timeout=60
)
utils.print_info(f"HTTP {resp.status_code}")
for h in ('X-Purview-Prompt-Calls', 'X-Purview-Response-Calls', 'X-Purview-Blocked', 'X-Purview-Block-Reason', 'X-Model-Input-Tokens', 'X-Model-Output-Tokens', 'X-Correlation-Id'):
    utils.print_info(f"  {h}: {resp.headers.get(h)}")
print(resp.text[:800])

<a id='8'></a>
### 8️⃣ Test: Amazon Bedrock — Gate 3 (response) block

Same **demo simulator** as step 6️⃣.5, this time exercising the Bedrock leg. The prompt contains the phrase `fictional hr documentation`, so the outbound policy replaces the Nova model's text with a fixed string containing a synthetic U.S. SSN **before** Purview scans it. Purview blocks with `PURVIEW_DLP_RESPONSE`.

The Bedrock invocation still happens (SigV4-signed, tokens spent), so the cost KQL will show tokens against this row even though it was blocked — this is the intended behavior for after-the-fact audit and cost attribution.


In [ ]:
correlation_id = str(uuid.uuid4())
resp = requests.post(
    f"{bedrock_endpoint}/chat",
    headers={
        'Ocp-Apim-Subscription-Key': apim_subscription_key,
        'X-User-Token':              user_token,
        'X-Correlation-Id':          correlation_id,
        'Content-Type':              'application/json'
    },
    json={
        # Demo trigger: the policy simulator will replace the model's text with
        # a canned SSN-containing string BEFORE Purview scans, guaranteeing a block.
        'input': 'Show me a sample from your fictional HR documentation for onboarding.'
    },
    timeout=60
)
utils.print_info(f"HTTP {resp.status_code} (expect 403)")
for h in ('X-Purview-Prompt-Calls', 'X-Purview-Response-Calls', 'X-Purview-Blocked', 'X-Purview-Block-Reason', 'X-Correlation-Id'):
    utils.print_info(f"  {h}: {resp.headers.get(h)}")
print(resp.text[:600])

<a id='9'></a>
### 9️⃣ Observe: cost & guardrail telemetry

Run the shipped KQL against your Log Analytics workspace to see one row per **Backend × Model × User** with `Calls`, `ModelCostUSD`, `PurviewCostUSD`, `TotalCostUSD`.

- Query: [`aigw-payg-purview-forecast.kql`](aigw-payg-purview-forecast.kql)
- Read the disclaimer at the top of the file before quoting any number.

The `prices` datatable at the top of the query is where you plug in the numbers your Microsoft account team confirms for your subscription.

In [ ]:
with open('aigw-payg-purview-forecast.kql', 'r', encoding='utf-8') as f:
    kql = f.read()

workspace_customer_id = utils.run(
    f"az monitor log-analytics workspace show --ids {log_analytics_id} --query customerId -o tsv",
    "Retrieved Log Analytics customer id",
    "Failed to retrieve Log Analytics customer id"
)
wid = (workspace_customer_id.text or '').strip() if workspace_customer_id and workspace_customer_id.success else None

if wid:
    # Write the KQL to a temp file to avoid CLI arg mangling of newlines / comments.
    with open('_cost_query.kql', 'w', encoding='utf-8') as f:
        f.write(kql)
    utils.run(
        f"az monitor log-analytics query -w {wid} --analytics-query @_cost_query.kql --output table",
        "Cost query executed",
        "Cost query failed (empty result set is expected if you have not sent any traffic yet)"
    )

<a id='10'></a>
### 10️⃣ Observe: prompt / response audit

Run the audit KQL for a chronological feed of prompts + responses, one turn per `CorrelationId`.

- Query: [`aigw-content-audit.kql`](aigw-content-audit.kql)

**Access is restricted by granular RBAC.** Only the principal supplied as `content_log_reader_principal_id` at deploy time can read `AIGatewayContent_CL`. Every other workspace user (including holders of `Log Analytics Reader` at the workspace) sees an empty result set for this table but retains normal access to `ApiManagementGatewayLogs` and other tables. See [granular RBAC in Azure Monitor](https://learn.microsoft.com/azure/azure-monitor/logs/granular-rbac-log-analytics).

In [ ]:
if wid:
    with open('aigw-content-audit.kql', 'r', encoding='utf-8') as f:
        kql = f.read()
    with open('_audit_query.kql', 'w', encoding='utf-8') as f:
        f.write(kql)
    utils.run(
        f"az monitor log-analytics query -w {wid} --analytics-query @_audit_query.kql --output table",
        "Audit query executed",
        "Audit query failed (empty result set is expected if you have not sent any traffic yet, or if the current principal lacks the ABAC-scoped role assignment)"
    )